# Experiment 8: LoRA Rank Ablation — Fixing the Fused QKV Capacity Gap

**Aim:** Test whether increasing LoRA rank on ModernBERT's fused `Wqkv` closes the ~4pp gap vs BERT-base.

**Hypothesis:** ModernBERT's fused QKV projection (`Wqkv`) gets a single rank-16 adapter shared across Q, K, V — effective rank ~5 per component. BERT's separate `query`, `key`, `value` each get rank-16 — 3× the attention LoRA capacity. Using r=48 on `Wqkv` should equalize this.

**Experiments:**
1. **ModernBERT + LoRA r=16** (baseline, same as NB01) — reference
2. **ModernBERT + LoRA r=48** on `Wqkv` only, r=16 on other modules
3. **ModernBERT + LoRA r=48** on all modules

**Comparison:** BERT-base + LoRA r=16 from NB05 (95.19% on FPB 50agree, 99.16% on allAgree)

**Training data:** Same aggregated dataset excluding FPB (source 5), same hyperparameters as NB01/NB05.

## 1. Setup & Installation

In [ ]:
# setup

In [ ]:
%%capture
!pip install -q "datasets>=3.4.1,<4.0.0" scikit-learn matplotlib seaborn peft accelerate transformers

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import torch
import torch.nn.functional as F
from transformers import (
    TrainingArguments, Trainer, AutoModelForSequenceClassification,
    AutoTokenizer, training_args,
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import gc

NUM_CLASSES = 3
LABEL_NAMES = ["NEGATIVE", "NEUTRAL", "POSITIVE"]
FPB_SOURCE = 5

## 2. Load Data

In [ ]:
# Load FPB test sets
fpb_50 = load_dataset("financial_phrasebank", "sentences_50agree", trust_remote_code=True)["train"]
fpb_all = load_dataset("financial_phrasebank", "sentences_allagree", trust_remote_code=True)["train"]

print(f"FPB 50agree: {len(fpb_50):,} samples")
print(f"FPB allAgree: {len(fpb_all):,} samples")

for i, name in enumerate(LABEL_NAMES):
    count_50 = sum(1 for l in fpb_50["label"] if l == i)
    count_all = sum(1 for l in fpb_all["label"] if l == i)
    print(f"  {name}: 50agree={count_50}, allAgree={count_all}")

In [ ]:
# Load aggregated training data (same as NB01/NB05)
ds = load_dataset("neoyipeng/financial_reasoning_aggregated")

label_dict = {"NEUTRAL/MIXED": 1, "NEGATIVE": 0, "POSITIVE": 2}

ds = ds.filter(lambda x: x["task"] == "sentiment")
ds = ds.filter(lambda x: x["source"] != FPB_SOURCE)

remove_cols = [c for c in ds["train"].column_names if c not in ("text", "labels")]
ds = ds.map(
    lambda ex: {
        "text": ex["text"],
        "labels": np.eye(NUM_CLASSES)[label_dict[ex["label"]]],
    },
    remove_columns=remove_cols,
)

print(f"Train: {len(ds['train']):,}  |  Val: {len(ds['validation']):,}  |  Test: {len(ds['test']):,}")

## 3. Evaluation Helper

In [ ]:
def evaluate_on_fpb(model, tokenizer, fpb_dataset, batch_size=32):
    """Evaluate a model on an FPB dataset split."""
    texts = fpb_dataset["sentence"]
    labels = fpb_dataset["label"]

    all_preds = []

    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Evaluating"):
            batch_texts = texts[i : i + batch_size]
            inputs = tokenizer(
                batch_texts, return_tensors="pt", padding=True,
                truncation=True, max_length=512,
            )
            inputs = {k: v.cuda() for k, v in inputs.items()}
            logits = model(**inputs).logits
            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            all_preds.extend(preds)

    y_true = np.array(labels)
    y_pred = np.array(all_preds)

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    report = classification_report(y_true, y_pred, target_names=LABEL_NAMES)
    cm = confusion_matrix(y_true, y_pred)

    print(f"Accuracy: {acc:.4f} ({int(acc * len(y_true))}/{len(y_true)})")
    print(f"Macro F1: {macro_f1:.4f}")
    print(f"\n{report}")

    return {"accuracy": acc, "macro_f1": macro_f1, "report": report, "cm": cm}

## 4. Training Function

In [ ]:
def train_and_evaluate(lora_config, run_name, output_dir):
    """Train ModernBERT with given LoRA config, evaluate on FPB."""
    print(f"\n{'='*70}")
    print(f"TRAINING: {run_name}")
    print(f"{'='*70}")

    model = AutoModelForSequenceClassification.from_pretrained(
        "answerdotai/ModernBERT-base",
        num_labels=NUM_CLASSES,
        torch_dtype=torch.float32,
        attn_implementation="sdpa",
    )
    model.gradient_checkpointing_enable()
    tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

    model = get_peft_model(model, lora_config)
    model = model.cuda()
    model.print_trainable_parameters()

    def tokenize_function(examples):
        return tokenizer(examples["text"])

    train_data = ds["train"].map(tokenize_function, batched=True)
    val_data = ds["validation"].map(tokenize_function, batched=True)

    trainer = Trainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=train_data,
        eval_dataset=val_data,
        args=TrainingArguments(
            output_dir=output_dir,
            per_device_train_batch_size=8,
            gradient_accumulation_steps=4,
            warmup_steps=10,
            fp16=True,
            bf16=False,
            optim=training_args.OptimizerNames.ADAMW_TORCH,
            learning_rate=2e-4,
            weight_decay=0.001,
            lr_scheduler_type="cosine",
            seed=3407,
            num_train_epochs=10,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            save_strategy="epoch",
            eval_strategy="epoch",
            logging_strategy="epoch",
            gradient_checkpointing=True,
            report_to="none",
        ),
        compute_metrics=lambda eval_pred: {
            "accuracy": accuracy_score(
                eval_pred[1].argmax(axis=-1), eval_pred[0].argmax(axis=-1)
            )
        },
    )

    trainer.train()
    model = model.cuda().eval()

    # Evaluate on aggregated test set
    test_texts = ds["test"]["text"]
    test_labels = np.argmax(ds["test"]["labels"], axis=1)
    all_test_preds = []
    with torch.no_grad():
        for i in tqdm(range(0, len(test_texts), 32), desc="Agg test"):
            batch = test_texts[i : i + 32]
            inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512)
            inputs = {k: v.cuda() for k, v in inputs.items()}
            logits = model(**inputs).logits
            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            all_test_preds.extend(preds)

    agg_acc = accuracy_score(test_labels, np.array(all_test_preds))
    agg_f1 = f1_score(test_labels, np.array(all_test_preds), average="macro")
    print(f"\n{run_name} Aggregated Test — Accuracy: {agg_acc:.4f}  Macro F1: {agg_f1:.4f}")
    print(classification_report(test_labels, np.array(all_test_preds), target_names=LABEL_NAMES))

    # Evaluate on FPB
    print(f"\n--- {run_name} — FPB sentences_50agree ---")
    res_50 = evaluate_on_fpb(model, tokenizer, fpb_50)

    print(f"\n--- {run_name} — FPB sentences_allAgree ---")
    res_all = evaluate_on_fpb(model, tokenizer, fpb_all)

    # Cleanup
    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "name": run_name,
        "agg_acc": agg_acc, "agg_f1": agg_f1,
        "fpb_50_acc": res_50["accuracy"], "fpb_50_f1": res_50["macro_f1"],
        "fpb_all_acc": res_all["accuracy"], "fpb_all_f1": res_all["macro_f1"],
        "cm_50": res_50["cm"], "cm_all": res_all["cm"],
        "report_50": res_50["report"], "report_all": res_all["report"],
    }

## 5. Run 1: ModernBERT + LoRA r=16 (Baseline)

In [ ]:
lora_r16 = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["Wqkv", "out_proj", "Wi", "Wo"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

results_r16 = train_and_evaluate(lora_r16, "ModernBERT r=16", "trainer_output_r16")

## 6. Run 2: ModernBERT + LoRA r=48 (Matching BERT Effective Capacity)

In [ ]:
lora_r48 = LoraConfig(
    r=48,
    lora_alpha=96,
    target_modules=["Wqkv", "out_proj", "Wi", "Wo"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

results_r48 = train_and_evaluate(lora_r48, "ModernBERT r=48", "trainer_output_r48")

## 7. Run 3: ModernBERT + LoRA r=48 on Wqkv Only, r=16 Elsewhere

Test whether the gain comes specifically from attention capacity (Wqkv) or from overall LoRA capacity.

In [ ]:
# PEFT doesn't support per-module rank natively, so we use rank_pattern
lora_r48_wqkv = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["Wqkv", "out_proj", "Wi", "Wo"],
    rank_pattern={"Wqkv": 48},
    alpha_pattern={"Wqkv": 96},
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

results_r48_wqkv = train_and_evaluate(lora_r48_wqkv, "ModernBERT r=48(Wqkv)/r=16", "trainer_output_r48_wqkv")

## 8. Comparison Table

In [ ]:
all_results = [results_r16, results_r48, results_r48_wqkv]

# Add NB05 BERT baseline for reference
comparison = pd.DataFrame([
    {"Model": "BERT-base + LoRA r=16 (NB05)",
     "FPB 50agree Acc": 0.9519, "FPB 50agree F1": 0.9446,
     "FPB allAgree Acc": 0.9916, "FPB allAgree F1": 0.9864,
     "Agg Test Acc": 0.8188, "Agg Test F1": 0.7787},
] + [
    {"Model": r["name"],
     "FPB 50agree Acc": r["fpb_50_acc"], "FPB 50agree F1": r["fpb_50_f1"],
     "FPB allAgree Acc": r["fpb_all_acc"], "FPB allAgree F1": r["fpb_all_f1"],
     "Agg Test Acc": r["agg_acc"], "Agg Test F1": r["agg_f1"]}
    for r in all_results
])

print("=" * 110)
print("LORA RANK ABLATION — ModernBERT vs BERT-base")
print("=" * 110)
print(comparison.to_string(index=False, float_format="%.4f"))

print("\n--- Delta vs BERT-base (NB05) ---")
bert_50 = 0.9519
for r in all_results:
    delta = r["fpb_50_acc"] - bert_50
    print(f"  {r['name']:35s}: {delta:+.4f} ({delta*100:+.2f}pp)")

## 9. Visualizations

In [ ]:
# Bar chart: FPB 50agree accuracy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models = ["BERT-base r=16\n(NB05)"] + [r["name"] for r in all_results]
accs_50 = [0.9519] + [r["fpb_50_acc"] for r in all_results]
accs_all = [0.9916] + [r["fpb_all_acc"] for r in all_results]
colors = ["#90CAF9"] + ["#FFA726", "#66BB6A", "#AB47BC"]

for ax, accs, title in [
    (axes[0], accs_50, "FPB sentences_50agree"),
    (axes[1], accs_all, "FPB sentences_allAgree"),
]:
    bars = ax.barh(models, accs, color=colors, edgecolor="white")
    ax.set_xlim(0.75, 1.0)
    ax.set_xlabel("Accuracy")
    ax.set_title(title)
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height() / 2,
                f"{acc:.1%}", va="center", fontsize=10)

plt.suptitle("LoRA Rank Ablation — FPB Accuracy", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("lora_rank_ablation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Confusion matrices for all ModernBERT variants on FPB 50agree
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, r in zip(axes, all_results):
    sns.heatmap(
        r["cm_50"], annot=True, fmt="d", cmap="Blues",
        xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax,
    )
    ax.set_title(f"{r['name']}\nAcc={r['fpb_50_acc']:.2%}  F1={r['fpb_50_f1']:.2%}")
    ax.set_ylabel("True")
    ax.set_xlabel("Predicted")

plt.suptitle("Confusion Matrices — FPB sentences_50agree", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("lora_rank_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Analysis

In [ ]:
print("=" * 70)
print("ANALYSIS")
print("=" * 70)

print("\n--- LoRA Rank Effect on ModernBERT ---")
print(f"  r=16 (baseline):           {results_r16['fpb_50_acc']:.4f} acc / {results_r16['fpb_50_f1']:.4f} F1")
print(f"  r=48 (all modules):        {results_r48['fpb_50_acc']:.4f} acc / {results_r48['fpb_50_f1']:.4f} F1")
print(f"  r=48 Wqkv / r=16 rest:     {results_r48_wqkv['fpb_50_acc']:.4f} acc / {results_r48_wqkv['fpb_50_f1']:.4f} F1")

print("\n--- Gap vs BERT-base (NB05: 95.19% / 94.46%) ---")
for r in all_results:
    gap_acc = 0.9519 - r["fpb_50_acc"]
    gap_f1 = 0.9446 - r["fpb_50_f1"]
    print(f"  {r['name']:35s}: {gap_acc:+.4f} acc / {gap_f1:+.4f} F1 (BERT advantage)")

best = max(all_results, key=lambda x: x["fpb_50_acc"])
print(f"\n--- Best ModernBERT variant: {best['name']} ---")
print(f"  FPB 50agree:  {best['fpb_50_acc']:.4f} acc / {best['fpb_50_f1']:.4f} F1")
print(f"  FPB allAgree: {best['fpb_all_acc']:.4f} acc / {best['fpb_all_f1']:.4f} F1")
print(f"  Remaining gap vs BERT: {0.9519 - best['fpb_50_acc']:.4f} acc")

if best["fpb_50_acc"] > results_r16["fpb_50_acc"]:
    improvement = best["fpb_50_acc"] - results_r16["fpb_50_acc"]
    total_gap = 0.9519 - results_r16["fpb_50_acc"]
    pct_closed = improvement / total_gap * 100 if total_gap > 0 else 0
    print(f"  Gap closed by rank increase: {improvement:.4f} ({pct_closed:.1f}% of original gap)")